In [3]:
# =========================================
# 1️⃣ Download Data
# =========================================

SYMBOL = "AAPL"
API_KEY = "167D13YZGB60IS2N"

df = download_alpha_vantage(SYMBOL, API_KEY)

print(df.head())

# =========================================
# 2️⃣ Build Alpha Components
# =========================================

# Returns
df["ret_1"] = df["close"] / df["close"].shift(1) - 1
df["ret_3"] = df["close"] / df["close"].shift(3) - 1

# 7-day volatility
df["vol_7"] = df["ret_1"].rolling(7).std()

# Intraday component
df["intraday"] = -1 * (2*df["close"] - df["high"] - df["low"]) / (df["high"] - df["low"])

# Relative volume
df["adv20"] = df["volume"].rolling(20).mean()
df["rel_vol"] = df["volume"] / df["adv20"]

# Drop NaNs
df = df.dropna()

print(df.tail())

# =========================================
# 3️⃣ Build Alpha Components
# =========================================

# --- Returns ---
df["ret_1"] = df["close"] / df["close"].shift(1) - 1
df["ret_3"] = df["close"] / df["close"].shift(3) - 1

# --- Volatility (7-day std of daily returns) ---
df["vol_7"] = df["ret_1"].rolling(7).std()

# --- ADV20 ---
df["adv20"] = df["volume"].rolling(20).mean()

# --- Component 1 ---
df["comp1"] = -df["ret_3"] / df["vol_7"]

# --- Component 2 ---
df["comp2"] = -((2*df["close"] - df["high"] - df["low"]) / (df["high"] - df["low"]))

# --- Volume Rank ---
df["vol_ratio"] = df["volume"] / df["adv20"]
df["rank_vol"] = df["vol_ratio"].rank(pct=True)

# --- Final Alpha ---
df["raw_alpha"] = (df["comp1"] + df["comp2"]) * df["rank_vol"]

# Final Rank
df["alpha"] = df["raw_alpha"].rank(pct=True)

df = df.dropna()

# =========================================
# 4️⃣ Simple Long/Short Backtest
# =========================================

# Position: top 30% long, bottom 30% short
df["position"] = 0
df.loc[df["alpha"] > 0.7, "position"] = 1
df.loc[df["alpha"] < 0.3, "position"] = -1

# Forward return
df["fwd_ret"] = df["ret_1"].shift(-1)

# Strategy return
df["strategy_ret"] = df["position"] * df["fwd_ret"]

df["equity"] = (1 + df["strategy_ret"]).cumprod()

# =========================================
# 5️⃣ Performance Metrics
# =========================================

sharpe = np.sqrt(252) * df["strategy_ret"].mean() / df["strategy_ret"].std()

cum_max = df["equity"].cummax()
drawdown = df["equity"]/cum_max - 1
max_dd = drawdown.min()

hit_rate = (df["strategy_ret"] > 0).mean()

print("===== PERFORMANCE =====")
print(f"Sharpe: {sharpe:.2f}")
print(f"Max Drawdown: {max_dd:.2%}")
print(f"Hit Rate: {hit_rate:.2%}")

# =========================================
# 6️⃣ Visualization Dashboard
# =========================================

fig, axes = plt.subplots(2,2, figsize=(14,8))

# Equity
axes[0,0].plot(df.index, df["equity"])
axes[0,0].set_title("Equity Curve")

# Drawdown
axes[0,1].plot(df.index, drawdown, color="red")
axes[0,1].set_title("Drawdown")

# Return Distribution
axes[1,0].hist(df["strategy_ret"], bins=50)
axes[1,0].set_title("Return Distribution")

# Alpha Signal
axes[1,1].plot(df.index, df["alpha"])
axes[1,1].set_title("Alpha Signal")

plt.tight_layout()
plt.show()


API Response Keys: dict_keys(['Information'])
⚠ API Error Response:
{'Information': 'Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium endpoints'}


ValueError: Alpha Vantage did not return price data. Check API limit or symbol.